# SmolVLA T4 replay

Real SmolVLA inference on a Colab T4, offline replay of the frozen corpus, Kineto trace + metrics + fingerprints per run, ingested by the existing Reflex pipeline. Until this work merges to main, set `REFLEX_REF` to the feature branch or an exact commit SHA (recorded below as COMMIT).

In [ ]:
# Cell 1 - T4 probe and run directory.
import subprocess
from pathlib import Path
probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], text=True, capture_output=True, check=True)
print(probe.stdout.strip())
if 'Tesla T4' not in probe.stdout and 'P100' not in probe.stdout:
    raise RuntimeError('Expected a T4/P100-or-better Colab GPU')
RUNS = Path('/content/reflex_runs')
RUNS.mkdir(exist_ok=True)
print('GPU probe passed')

In [ ]:
# Cell 2 - checkout the exact ref, install the pinned SmolVLA stack.
import os, subprocess, sys
from pathlib import Path
REPO = Path('/content/reflex')
REMOTE = 'https://github.com/MugiZer/reflex.git'
REFLEX_REF = os.environ.get('REFLEX_REF', 'workloads/smolvla-replay')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REMOTE, str(REPO)], check=True)
def git(*args):
    return subprocess.run(['git', *args], cwd=REPO, check=True, text=True, capture_output=True).stdout.strip()
git('fetch', 'origin', REFLEX_REF, '--prune')
target = f'origin/{REFLEX_REF}' if '/' not in REFLEX_REF and len(REFLEX_REF) != 40 else REFLEX_REF
git('checkout', '--detach', target)
COMMIT = git('rev-parse', 'HEAD')
print('testing commit:', COMMIT)
subprocess.run(['apt', 'install', '-y', '-q', 'ffmpeg'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--index-url', 'https://download.pytorch.org/whl/cu128', 'torch==2.9.1', 'torchvision==0.24.1'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lerobot[smolvla,dataset,evaluation]==0.6.0'], check=True)
os.chdir(REPO)

In [ ]:
# Cell 3 - import probe (no hardware extras) + read dataset meta.
from collections import Counter
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.datasets.lerobot_dataset import LeRobotDataset
DATASET, DATASET_REV = 'lerobot/svla_so100_pickplace', '728583b5eaf9e739a7f119e2def466fa1d552402'
CHECKPOINT, CHECKPOINT_REV = 'lerobot/smolvla_base', 'c83c3163b8ca9b7e67c509fffd9121e66cb96205'
ds = LeRobotDataset(DATASET, revision=DATASET_REV)
counts = Counter(int(e) for e in ds.hf_dataset['episode_index'])
lengths = [counts[e] for e in sorted(counts)]
assert len(lengths) == 50, lengths
print('episodes:', len(lengths), 'frames:', sum(lengths))
INSTRUCTION = str(ds.meta.tasks.index[0])
print('instruction:', repr(INSTRUCTION))
PINS = {'resize_pad': [512, 512], 'normalization': 'MEAN_STD-shipped', 'tokenizer': 'SmolVLM2-500M-Video-Instruct', 'max_length': 48, 'task_key': 'task', 'image_source_keys': ['observation.images.top', 'observation.images.wrist']}
print('import probe passed')

In [ ]:
# Cell 4 - matrix + corpus genesis (once) or frozen-consume (later).
import hashlib, json, os
from reflex import collect as C
from reflex.collect import nvidia_smi_identity
from workloads.smolvla import build_corpus as B, validate_corpus as V
ident = nvidia_smi_identity()
print('identity:', ident)
if ident['hardware'] == 'unknown':
    raise RuntimeError('nvidia-smi did not provide hardware identity')
CORPUS = REPO / 'workloads' / 'smolvla' / 'corpora'
if not (CORPUS / 'main-1000.jsonl').exists():
    (CORPUS / 'lengths.json').write_text(json.dumps(lengths))
    (CORPUS / 'preprocessing.json').write_text(json.dumps(PINS))
    bundle = B.build(lengths, INSTRUCTION, PINS)
    print(B.write_files(bundle, CORPUS))
    print('GENESIS manifests written — commit them as a reviewed diff before trusting later runs')
errs = V.validate(CORPUS)
assert not errs, errs
CORPUS_SHA = hashlib.sha256((CORPUS / 'main-1000.jsonl').read_bytes()).hexdigest()
print('corpus valid, sha:', CORPUS_SHA[:12])
faults = ('healthy',)
seeds = tuple(int(x) for x in os.environ.get('REFLEX_SEEDS', '11,17,23').split(','))
ROOT = RUNS / COMMIT[:12]
DATASET_OUT = ROOT / 'smolvla-dataset.jsonl'
TIERS = {'smoke': 'smolvla-smoke-v1', 'main': 'smolvla-replay-v1'}
print({'seeds': seeds, 'corpus': CORPUS_SHA[:12], 'root': str(ROOT)})

In [ ]:
# Cell 5 - smoke tier first, then the main corpus. Same device seam.
import torch
from workloads.smolvla.replay import make_device
import lerobot
SOFTWARE = {'torch': torch.__version__, 'lerobot': lerobot.__version__, 'checkpoint': CHECKPOINT, 'checkpoint_rev': CHECKPOINT_REV, 'dataset': DATASET, 'dataset_rev': DATASET_REV, 'corpus_sha256': CORPUS_SHA, 'rng': 0, 'warmup': 'holdout-5x2', 'dtype': 'float32', 'video_backend': 'pyav', 'repeat_is_seed': 'manifest seed is repeat identity; model RNG fixed at 0'}
results = {}
for tier, workload in [('smoke', TIERS['smoke']), ('main', TIERS['main'])]:
    frames_file = 'smoke-250.jsonl' if tier == 'smoke' else 'main-1000.jsonl'
    device = make_device(CORPUS, CHECKPOINT, CHECKPOINT_REV, DATASET, DATASET_REV, frames_file=frames_file)
    target = [('healthy', ident['hardware'], C.COLLECTOR_VERSION)]
    pipe = C.run_pipeline(ROOT / tier, DATASET_OUT, target, faults=faults, seeds=seeds, device=device, identity_provider=nvidia_smi_identity, workload=workload, trace_variant='kineto', perf_status='profiled', torch_version=torch.__version__, software=SOFTWARE, commit=COMMIT)
    results[tier] = pipe
    print(tier, json.dumps({'failed': pipe['collected']['failed'], 'records': pipe['records'], 'gaps': pipe['gaps']}, default=str))
print('tiers done:', sorted(results))

In [ ]:
# Cell 6 - publish commit, corpus, tiers, failures, gaps.
import datetime, uuid
from colab.report_results import publish
result = {'run_id': datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8], 'status': 'passed' if all(not r['collected']['failed'] and not r['gaps'] for r in results.values()) else 'incomplete', 'commit': COMMIT, 'requested_ref': REFLEX_REF, 'hardware': ident, 'corpus_sha256': CORPUS_SHA, 'matrix': {'tiers': TIERS, 'faults': faults, 'seeds': seeds}, 'pipeline': results}
Path('/content/reflex_runs/run_result.json').write_text(json.dumps(result, indent=2, default=str), encoding='utf-8')
publish(result)
print(json.dumps({'status': result['status']}, indent=2))

In [ ]:
# Cell 7 - optional Drive backup. Copy only after checksums and ingest.
try:
    from google.colab import drive
    import datetime
    drive.mount('/content/drive')
    dst = '/content/drive/MyDrive/reflex-colab-t4/%s-smolvla' % datetime.date.today().isoformat()
    import shutil
    shutil.copytree('/content/reflex_runs', dst, dirs_exist_ok=True)
    print('backed up to', dst)
except Exception as exc:
    print('drive backup skipped:', type(exc).__name__, exc)